In [2]:
!pip install matplotlib

import os
import pandas as pd
import matplotlib.pyplot as plt
import psycopg2

conn = psycopg2.connect(
    host=os.getenv("PGHOST", "db"),
    port=int(os.getenv("PGPORT", "5432")),
    dbname=os.getenv("PGDATABASE", "zava"),
    user=os.getenv("PGUSER", "postgres"),
    password=os.getenv("PGPASSWORD", "P@ssw0rd!"),
)

query = """
WITH monthly AS (
    SELECT
        c.category_name,
        EXTRACT(MONTH FROM o.order_date)::int AS month_num,
        SUM(oi.quantity) AS units_sold
    FROM retail.orders o
    JOIN retail.order_items oi
      ON o.order_id = oi.order_id
    JOIN retail.products p
      ON oi.product_id = p.product_id
    JOIN retail.categories c
      ON p.category_id = c.category_id
    GROUP BY c.category_name, EXTRACT(MONTH FROM o.order_date)
)
SELECT category_name, month_num, units_sold
FROM monthly
ORDER BY category_name, month_num
"""

monthly = pd.read_sql(query, conn)
conn.close()

summary_rows = []
for category, group in monthly.groupby("category_name", sort=True):
    if group.empty:
        continue

    peak_row = group.loc[group["units_sold"].idxmax()]
    low_row = group.loc[group["units_sold"].idxmin()]

    peak_units = float(peak_row["units_sold"])
    low_units = float(low_row["units_sold"])

    summary_rows.append({
        "Category": category,
        "Peak Month": int(peak_row["month_num"]),
        "Low Month": int(low_row["month_num"]),
        "Peak Units": round(peak_units, 2),
        "Low Units": round(low_units, 2),
        "Swing": round(peak_units - low_units, 2),
        "Peak/Low Ratio": round(peak_units / low_units, 2) if low_units else None,
    })

summary = pd.DataFrame(summary_rows).sort_values("Swing", ascending=False).reset_index(drop=True)
display(summary.head(10))

# Optional chart from the database data
chart = summary.head(10)
plt.figure(figsize=(10, 6))
plt.bar(chart["Category"], chart["Swing"], color="steelblue")
plt.title("Top Seasonal Swing by Category")
plt.xlabel("Category")
plt.ylabel("Swing (Units)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


,Category,Peak Month,Low Month,Peak Multiplier,Low Multiplier,Swing
0,GARDEN & OUTDOOR,4,2,2.5,0.5,2.0
1,LUMBER & BUILDING MATERIALS,6,2,2.2,0.7,1.5
2,POWER TOOLS,7,12,2.1,0.8,1.3
3,PAINT & FINISHES,4,11,2.2,1.0,1.2
4,STORAGE & ORGANIZATION,12,6,1.8,0.8,1.0
5,HARDWARE,10,1,1.8,1.0,0.8
6,HAND TOOLS,5,12,1.6,0.9,0.7
7,ELECTRICAL,11,3,1.6,1.0,0.6
8,PLUMBING,10,2,1.6,1.0,0.6


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

bars = plt.bar(summary['Category'], summary['Swing'], color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22'])
plt.title('Seasonal Swing by Category')
plt.xlabel('Category')
plt.ylabel('Swing')
plt.xticks(rotation=25, ha='right')
plt.grid(axis='y', alpha=0.3)

for bar, value in zip(bars, summary['Swing']):
    plt.text(bar.get_x() + bar.get_width() / 2, value + 0.05, f'{value:.2f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()